# Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import json
from scipy.optimize import linprog

# Load the data

In [8]:
df_risk = pd.read_csv('cavite_drought_risk_coefficients.csv')
municipalities = df_risk['Municipality'].values
R = df_risk['risk_score'].values
n = len(municipalities)

# LPP Formulation

Let the operational parameters and regional topology of Cavite be defined as follows:

* $\mathcal{M} = \{1, 2, \dots, 23\}$: The index set denoting all 23 local government units ($16$ municipalities and $7$ component cities) of Cavite ($|\mathcal{M}| = 23$).
* $\mathcal{U} \subset \mathcal{M}$: The subset containing the nine ($9$) designated upland agricultural LGUs ($|\mathcal{U}| = 9$), leaving $14$ lowland LGUs in $\mathcal{M} \setminus \mathcal{U}$.
* $d_i \in \{0, 1\}$: A deterministic parameter encoding regional agricultural zoning:
$$
\begin{equation*}
d_i = \begin{cases} 
1, & \text{if } i \in \mathcal{U} \\ 
0, & \text{if } i \in \mathcal{M} \setminus \mathcal{U} 
\end{cases}
\end{equation*}
$$
* $R_i \in [0, 1]$: The empirical drought risk score of LGU $i$, defined by the complement of predicted root-zone soil wetness: $R_i = 1 - \overline{\text{GWETROOT}}_i$.
* $B = \text{PHP } 35{,}000{,}000$: The total provincial statutory preparedness budget ceiling.
* $L_i = \text{PHP } 400{,}000$: The mandatory operational survival floor for each LGU $i \in \mathcal{M}$.
* $U_i = 0.15 \, B = \text{PHP } 5{,}250{,}000$: The individual equity absorption ceiling for each LGU $i \in \mathcal{M}$.

The resource distribution model is formulated as the following continuous Linear Programming Problem (LPP):
$$
\begin{equation*}
\begin{aligned}
\text{\textbf{maximize}} \quad & Z = \sum_{i \in \mathcal{M}} R_i x_i \\[0.5em]
\text{\textbf{subject to:}} \quad 
& \sum_{i \in \mathcal{M}} x_i \le B && \text{(Statutory Preparedness Budget Ceiling)} \\
& \sum_{i \in \mathcal{M}} d_i x_i \ge 0.60 \, B && \text{(Zonal Upland Agricultural Guarantee)} \\
& x_i \ge L_i, \quad \forall i \in \mathcal{M} && \text{(Operational Survival Floor)} \\
& x_i \le U_i, \quad \forall i \in \mathcal{M} && \text{(Equity Absorption Ceiling)} \\
& x_i \ge 0, \quad \forall i \in \mathcal{M} && \text{(Non-Negativity)}
\end{aligned}
\end{equation*}
$$
where each decision variable $x_i \in \mathbb{R}_{\ge 0}$ denotes the continuous fiscal appropriation granted to municipality or city $i$.

# Set the Baseline Parameters

In [3]:
TOTAL_LDRRMF = 50_000_000.0
BUDGET = 0.70 * TOTAL_LDRRMF  # PHP 35,000,000
FLOOR = 400_000.0  # PHP 400,000
CEILING = 0.15 * BUDGET  # PHP 5,250,000

## Matrix Representation

In [4]:
A_ub = []
b_ub = []
A_ub.append(np.ones(n))

#Constraint 1: Global Budget (sum x_i <= BUDGET)
b_ub.append(BUDGET)

# Upland Cluster Equity 
* (sum_{Upland} x_i >= 0.60 * BUDGET)

In [5]:
upland_set = {
    'Tagaytay City',
    'Silang',
    'Alfonso',
    'Amadeo',
    'Indang',
    'Mendez',
    'Magallanes',
    'General Emilio Aguinaldo',
    'Maragondon',
}

row_upland = np.zeros(n)
for i, muni in enumerate(municipalities):
  if muni in upland_set:
    row_upland[i] = -1.0  # Negated to fit <= form

A_ub.append(row_upland)
b_ub.append(-0.60 * BUDGET)

A_ub = np.array(A_ub)
b_ub = np.array(b_ub)

# Solve the LPP

In [6]:
## Execute Dantzig's Simplex Algorithm via HiGHS
res = linprog(c=-R, A_ub=A_ub, b_ub=b_ub, bounds=[(FLOOR, CEILING) for _ in range(n)], method='highs')

## Tabulate Allocation Results

In [7]:
if res.success:
  df_risk['Allocated_PHP'] = res.x
  df_risk['Pct_of_Budget'] = (res.x / BUDGET) * 100.0
  df_risk['Is_Upland'] = df_risk['Municipality'].isin(upland_set)

  print('Optimization Solved Successfully!')
  print(f'Total Budget Distributed: PHP {res.x.sum():,.2f}')
  print(f'Optimal Objective Value (Z*): {-res.fun:,.4f}\n')

  summary_cols = [
      'Municipality',
      'Is_Upland',
      'risk_score',
      'Allocated_PHP',
      'Pct_of_Budget',
  ]
  print(
      df_risk[summary_cols]
      .sort_values(by='Allocated_PHP', ascending=False)
      .to_string(index=False)
  )

  # Validate Zonal Equity
  upland_total = df_risk[df_risk['Is_Upland']]['Allocated_PHP'].sum()
  print(
      f'\nUpland Allocation: PHP {upland_total:,.2f} ({(upland_total/BUDGET)*100:.2f}%)'
  )
else:
  print(f'Optimization Failed: {res.message}')

Optimization Solved Successfully!
Total Budget Distributed: PHP 35,000,000.00
Optimal Objective Value (Z*): 15,762,541.9048

            Municipality  Is_Upland  risk_score  Allocated_PHP  Pct_of_Budget
                  Mendez       True    0.478793      5250000.0      15.000000
General Emilio Aguinaldo       True    0.478793      5250000.0      15.000000
                  Indang       True    0.478793      5250000.0      15.000000
                  Amadeo       True    0.478793      5250000.0      15.000000
              Magallanes       True    0.478793      5250000.0      15.000000
                 Alfonso       True    0.478793      1950000.0       5.571429
                 Ternate      False    0.342929       400000.0       1.142857
                   Tanza      False    0.342929       400000.0       1.142857
           Tagaytay City       True    0.287799       400000.0       1.142857
                  Silang       True    0.287799       400000.0       1.142857
                 

# Scenario Analysis: 

* Official DOST-PAGASA Climate Advisory (Press Release No. 225) warning of an
intensifying “Very Strong” El Niño event

### Rerun the Optimization algorithm

In [56]:
df_pert = pd.read_csv('cavite_drought_risk_coefficients_perturbed.csv')
municipalities = df_pert['Municipality'].values
R = df_pert['risk_score'].values
n = len(municipalities)

A_ub_pert = []
A_ub_pert.append(np.ones(n))

A_ub_pert.append(row_upland)

A_ub_pert = np.array(A_ub_pert)

res_pert = linprog(c=-R, A_ub=A_ub_pert, b_ub=b_ub, bounds=[(FLOOR, CEILING) for _ in range(n)], method='highs')

In [58]:
if res_pert.success:
  df_pert['Allocated_PHP'] = res_pert.x
  df_pert['Pct_of_Budget'] = (res_pert.x / BUDGET) * 100.0
  df_pert['Is_Upland'] = df_pert['Municipality'].isin(upland_set)

  print('Optimization Solved Successfully!')
  print(f'Total Budget Distributed: PHP {res_pert.x.sum():,.2f}')
  print(f'Optimal Objective Value (Z*): {-res_pert.fun:,.4f}\n')

  summary_cols = [
      'Municipality',
      'Is_Upland',
      'risk_score',
      'Allocated_PHP',
      'Pct_of_Budget',
  ]
  print(
      df_pert[summary_cols]
      .sort_values(by='Allocated_PHP', ascending=False)
      .to_string(index=False)
  )

  # Validate Zonal Equity
  upland_total = df_pert[df_pert['Is_Upland']]['Allocated_PHP'].sum()
  print(
      f'\nUpland Allocation: PHP {upland_total:,.2f} ({(upland_total/BUDGET)*100:.2f}%)'
  )
else:
  print(f'Optimization Failed: {res_pert.message}')

Optimization Solved Successfully!
Total Budget Distributed: PHP 35,000,000.00
Optimal Objective Value (Z*): 15,762,541.9048

            Municipality  Is_Upland  risk_score  Allocated_PHP  Pct_of_Budget
                  Mendez       True    0.478793      5250000.0      15.000000
General Emilio Aguinaldo       True    0.478793      5250000.0      15.000000
                  Indang       True    0.478793      5250000.0      15.000000
                  Amadeo       True    0.478793      5250000.0      15.000000
              Magallanes       True    0.478793      5250000.0      15.000000
                 Alfonso       True    0.478793      1950000.0       5.571429
                 Ternate      False    0.342929       400000.0       1.142857
                   Tanza      False    0.342929       400000.0       1.142857
           Tagaytay City       True    0.287799       400000.0       1.142857
                  Silang       True    0.287799       400000.0       1.142857
                 